# Path Length Tests

How does noise **really** affect the path towards converging to a stable state in the matrices?
In this notebook, we explore typical noiseless paths throughout development, as well as noisy
cases for some matrices.

In [ ]:
using LinearAlgebra, Distributions, Random, Plots, StatsBase, LaTeXStrings, JLD2, Base.Threads

const ROOT_DIR = normpath(joinpath(pwd(), "..", ".."))
const SRC_DIR = joinpath(ROOT_DIR, "src")
const SCRATCH_DIR = mktempdir()

include(joinpath(SRC_DIR, "wagner_algorithm.jl"))
include(joinpath(SRC_DIR, "data_processing.jl"))

const BN = BooleanNetwork
const Stats = CustomStats

Plots.theme(:dao)

println("Packages Loaded!")

In [ ]:
function explicit_develop(W, initial_state, max_steps;
    buffer1::Vector{Float64}=Vector{Float64}(undef, size(W, 1)),
    buffer2::Vector{Float64}=Vector{Float64}(undef, size(W, 1)),
    override_stability=false)    

    buffer1 .= initial_state
    buffer2 .= initial_state

    visited_states = Array{Vector{Float64}}(undef, max_steps+1)
    visited_states[1] = copy(initial_state)
    
    for step in 2:max_steps+1
        mul!(buffer2, W, buffer1)
        buffer2 .= sign.(buffer2)

        visited_states[step] = copy(buffer2)

        if buffer2 == buffer1
            # Stability reached: fill the remaining entries with the stable state
            for s in step+1:max_steps+1
                visited_states[s] = copy(buffer2)
            end
            return visited_states
        end

        buffer1, buffer2 = buffer2, buffer1  
    end
    if override_stability
        return visited_states
    else 
        return nothing
    end
end

function turn_into_distances(array_vectors, target_phenotype)
    return [sum(abs.(vec - target_phenotype)) for vec in array_vectors]
end

function plot_distances(main_distance, main_label,
    array_distances, array_label)
    p1 = plot(
        main_distance, 
        majorgrid=false,
        minorgrid=false,
        grid=false,
        xlabel="Development Time (t)",
        ylabel="Distance to Optimal " * L"\Vert P_{opt} - P(t) \Vert_1",
        label = main_label,
        color=:black,
        linewidth=1.5
    )

    for (idx, distance) in enumerate(array_distances)
        if idx == 1
            plot!(p1, distance, label=array_label, alpha=0.1, color=:gray)
        else
            plot!(p1, distance, label="", alpha=0.1, color=:gray)
        end
    end

    return p1
end


In [ ]:
# Shared parameters

number_genes = 10
pop_size = 200
weight_dist = Normal(0.0, 1.0)
number_paths = 1000
max_steps = 30;

## Random matrices

### Noiseless development path

We begin by generating a random stable matrix and seeing its development towards with reference to
an arbitrary optimal phenotype.

In [ ]:
simulation_params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=1
    )

initial_state, target_phenotype, matrices = BN.initialize_population(simulation_params)
W = matrices[1]

full_path = explicit_develop(W, initial_state, max_steps)

distances = turn_into_distances(full_path, target_phenotype);

In [ ]:
p1 = plot_distances(distances, "Noiseless", [], "")
plot!(p1, legend=:topright, title="Noiseless Development")
display(p1)

### Noisy Paths

In [ ]:
noise_dist = Gamma(1.0,1.0)

noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
noisy_paths = filter(!isnothing, explicit_develop.(noisy_matrices, Ref(initial_state), Ref(max_steps)))
noisy_distances = turn_into_distances.(noisy_paths, Ref(target_phenotype))
average_noisy_dist = mean(noisy_distances, dims = 1);

In [ ]:
p2 = plot_distances(average_noisy_dist, "Average", noisy_distances, "Individual Paths ")
plot!(p2, legend=:topright, title="Noisy Development - $(round(100* length(noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p2)

### Very Noisy Paths

In [ ]:
noise_dist = Gamma(1.0/8.0,8.0)

very_noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
very_noisy_paths = filter(!isnothing, explicit_develop.(very_noisy_matrices, Ref(initial_state), Ref(max_steps)))
very_noisy_distances = turn_into_distances.(very_noisy_paths, Ref(target_phenotype))
average_very_noisy_dist = mean(very_noisy_distances, dims = 1);

In [ ]:
p3 = plot_distances(average_very_noisy_dist, "Average", very_noisy_distances, "Individual Paths")
plot!(p3, legend=:topright, title="Very Noisy Development - $(round(100* length(very_noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p3)

In [ ]:
p_combined = plot(p1, p2, p3, layout=(1, 3), size=(1600, 400), 
    ylabel=["Distance to Optimal " * L"\Vert P_{opt} - P(t) \Vert_1" "" ""],
    left_margin=10Plots.mm)
savefig(p_combined, "random_combined_development.png")
display(p_combined)

## Evolved matrices - Noiseless

### Noiseless development path

We now generate a population of evolved matrices. We take one of them and observe its development
using its initial state and target phenotype. 

In [ ]:
params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist # Noiseless evolution
)

results = BN.run_simulation(params)
initial_state = results.initial_state
optimal_phenotype = results.optimal_phenotype
W = results.matrices_history[end,1]; # pick a single matrix

In [ ]:
full_path = explicit_develop(W, initial_state, max_steps)
distances = turn_into_distances(full_path, optimal_phenotype);

In [ ]:
p1 = plot_distances(distances, "Noiseless", [], "")
plot!(p1, legend=:topright, title="Noiseless Development")
display(p1)

### Noisy Paths

In [ ]:
noise_dist = Gamma(1.0,1.0)

noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
noisy_paths = filter(!isnothing, explicit_develop.(noisy_matrices, Ref(initial_state), Ref(max_steps)))
noisy_distances = turn_into_distances.(noisy_paths, Ref(optimal_phenotype))
average_noisy_dist = mean(noisy_distances, dims = 1);

In [ ]:
p2 = plot_distances(average_noisy_dist, "Average", noisy_distances, "Individual Paths")
plot!(p2, legend=:topright, title="Noisy Development - $(round(100* length(noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p2)

### Very Noisy Paths

In [ ]:
noise_dist = Gamma(1.0/8.0,8.0)

very_noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
very_noisy_paths = filter(!isnothing, explicit_develop.(very_noisy_matrices, Ref(initial_state), Ref(max_steps)))
very_noisy_distances = turn_into_distances.(very_noisy_paths, Ref(optimal_phenotype))
average_very_noisy_dist = mean(very_noisy_distances, dims = 1);

In [ ]:
p3 = plot_distances(average_very_noisy_dist, "Average", very_noisy_distances, "Individual Paths")
plot!(p3, legend=:topright, title="Very Noisy Development - $(round(100* length(very_noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p3)

In [ ]:
p_combined = plot(p1, p2, p3, layout=(1, 3), size=(1600, 400),
    ylabel=["Distance to Optimal " * L"\Vert P_{opt} - P(t) \Vert_1" "" ""],
    left_margin=10Plots.mm, suptitle = "Evolved Without Noise")
savefig(p_combined, "noiseless_combined_development.png")
display(p_combined)

## Evolved matrices - Noisy

### Noiseless development path

We now generate a population of evolved matrices, that experienced mild noise throughout evolution. 
We take one of them and observe its development using its initial state and target phenotype. 

In [ ]:
params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist, noise_dist=Gamma(1.0,1.0) # Noiseless evolution
)

results = BN.run_simulation(params)
initial_state = results.initial_state
optimal_phenotype = results.optimal_phenotype
W = results.matrices_history[end,1]; # pick a single matrix

In [ ]:
full_path = explicit_develop(W, initial_state, max_steps)
distances = turn_into_distances(full_path, optimal_phenotype);

In [ ]:
p1 = plot_distances(distances, "Noiseless", [], "")
plot!(p1, legend=:topright, title="Noiseless Development")
display(p1)

### Noisy Paths

In [ ]:
noise_dist = Gamma(1.0,1.0)

noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
noisy_paths = filter(!isnothing, explicit_develop.(noisy_matrices, Ref(initial_state), Ref(max_steps)))
noisy_distances = turn_into_distances.(noisy_paths, Ref(optimal_phenotype))
average_noisy_dist = mean(noisy_distances, dims = 1);

In [ ]:
p2 = plot_distances(average_noisy_dist, "Average", noisy_distances, "Individual Paths")
plot!(p2, legend=:topright, title="Noisy Development - $(round(100* length(noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p2)

### Very Noisy Paths

In [ ]:
noise_dist = Gamma(1.0/8.0,8.0)

very_noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
very_noisy_paths = filter(!isnothing, explicit_develop.(very_noisy_matrices, Ref(initial_state), Ref(max_steps)))
very_noisy_distances = turn_into_distances.(very_noisy_paths, Ref(optimal_phenotype))
average_very_noisy_dist = mean(very_noisy_distances, dims = 1);

In [ ]:
p3 = plot_distances(average_very_noisy_dist, "Average", very_noisy_distances, "Individual Paths")
plot!(p3, legend=:topright, title="Very Noisy Development - $(round(100* length(very_noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p3)

In [ ]:
p_combined = plot(p1, p2, p3, layout=(1, 3), size=(1600, 400),
    ylabel=["Distance to Optimal " * L"\Vert P_{opt} - P(t) \Vert_1" "" ""],
    left_margin=10Plots.mm, suptitle = "Evolved With Moderate Noise")
savefig(p_combined, "noisy_combined_development.png")
display(p_combined)

## Evolved matrices - Very Noisy

### Noiseless development path

We now generate a population of evolved matrices, that experienced high noise throughout evolution. 
We take one of them and observe its development using its initial state and target phenotype. 

In [ ]:
params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist, noise_dist=Gamma(1.0/8.0,8.0) # Very noisy evolution
)

results = BN.run_simulation(params)
initial_state = results.initial_state
optimal_phenotype = results.optimal_phenotype
W = results.matrices_history[end,1]; # pick a single matrix

In [ ]:
full_path = explicit_develop(W, initial_state, max_steps)
distances = turn_into_distances(full_path, optimal_phenotype);

In [ ]:
p1 = plot_distances(distances, "Noiseless", [], "")
plot!(p1, legend=:topright, title="Noiseless Development")
display(p1)

### Noisy Paths

In [ ]:
noise_dist = Gamma(1.0,1.0)

noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
noisy_paths = filter(!isnothing, explicit_develop.(noisy_matrices, Ref(initial_state), Ref(max_steps)))
noisy_distances = turn_into_distances.(noisy_paths, Ref(optimal_phenotype))
average_noisy_dist = mean(noisy_distances, dims = 1);

In [ ]:
p2 = plot_distances(average_noisy_dist, "Average", noisy_distances, "Individual Paths")
plot!(p2, legend=:topright, title="Noisy Development - $(round(100* length(noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p2)

### Very Noisy Paths

In [ ]:
noise_dist = Gamma(1.0/8.0,8.0)

very_noisy_matrices = [W.*rand(noise_dist,(number_genes,number_genes)) for _ in 1:number_paths]
very_noisy_paths = filter(!isnothing, explicit_develop.(very_noisy_matrices, Ref(initial_state), Ref(max_steps)))
very_noisy_distances = turn_into_distances.(very_noisy_paths, Ref(optimal_phenotype))
average_very_noisy_dist = mean(very_noisy_distances, dims = 1);

In [ ]:
p3 = plot_distances(average_very_noisy_dist, "Average", very_noisy_distances, "Individual Paths")
plot!(p3, legend=:topright, title="Very Noisy Development - $(round(100* length(very_noisy_paths) / number_paths,  digits = 4)) % Completion")
display(p3)

In [ ]:
p_combined = plot(p1, p2, p3, layout=(1, 3), size=(1600, 400),
    ylabel=["Distance to Optimal " * L"\Vert P_{opt} - P(t) \Vert_1" "" ""],
    left_margin=10Plots.mm, suptitle = "Evolved with High Noise")
savefig(p_combined, "very_noisy_combined_development.png")
display(p_combined)

## Response to Reviewer 1

In [ ]:
conditions = [
    (name=:noiseless, noise=nothing),
    (name=:noisy, noise=Gamma(1.0, 1.0)),
    (name=:very_noisy, noise=Gamma(1.0/8.0, 8.0)),
]

function run_condition(c)
    kwargs = (number_genes=number_genes, pop_size=pop_size,
        initial_pop_type=:stable, weights_dist=weight_dist)
    params = isnothing(c.noise) ?
             BN.SimulationParameters(; kwargs...) :
             BN.SimulationParameters(; kwargs..., noise_dist=c.noise)

    results = BN.run_simulation(params)
    W = results.matrices_history[end, 1]   # pick a single matrix
    path = explicit_develop(W, results.initial_state, max_steps)

    return (W=W,
        initial_state=results.initial_state,
        optimal_phenotype=results.optimal_phenotype,
        path=path,
        distances=turn_into_distances(path, results.optimal_phenotype))
end

outputs = Vector{Any}(undef, length(conditions))
for i in eachindex(conditions)
    outputs[i] = run_condition(conditions[i])
end

data = Dict(c.name => outputs[i] for (i, c) in enumerate(conditions));

In [ ]:
p1 = plot_distances(data[:noiseless][:distances], "Noiseless", [], "")
plot!(p1, legend=:false, title="A) Evolved without Noise")

p2 = plot_distances(data[:noisy][:distances], "Noiseless", [], "")
plot!(p2, legend=:false, title="B) Evolved with Moderate Noise " * L"\theta = 1.0")

p3 = plot_distances(data[:very_noisy][:distances], "Noiseless", [], "")
plot!(p3, legend=:false, title="C) Evolved with High Noise " * L"\theta = 8.0")

p_combined = plot(p1, p2, p3; layout = (1,3), size = (1600,400), dpi=900, margins = 8Plots.mm)
savefig(p_combined, "response_noiseless_dev.png")
display(p_combined)

# Figure 1.C Generator

In [ ]:
function wrapper_simulation(parameters)
    results = BN.run_simulation(parameters)
    initial_state = results.initial_state
    optimal_phenotype = results.optimal_phenotype
    W = results.matrices_history[end, 1]
    return W, initial_state, optimal_phenotype
end

In [ ]:
number_paths = 30
# Obtain a matrix evolved without noise
noiseless_params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist # Noiseless evolution
)
W_noiseless, init_noiseless, opt_noiseless = wrapper_simulation(noiseless_params)

# Obtain a matrix evolved with noise
noisy_params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist, noise_dist=Gamma(1.0,1.0) # Noiseless evolution
)
W_noisy, init_noisy, opt_noisy = wrapper_simulation(noisy_params)

# Obtain a matrix evolved with a lot of noise
very_noisy_params = BN.SimulationParameters(
    number_genes=number_genes, pop_size=pop_size, initial_pop_type=:stable,
    weights_dist=weight_dist, noise_dist=Gamma(1.0/8.0, 8.0) # Noiseless evolution
)
W_very_noisy, init_very_noisy, opt_very_noisy = wrapper_simulation(very_noisy_params)

# Generate paths for all Ws
all_Ws = [W_noiseless, W_noisy, W_very_noisy]
init_states = [init_noiseless, init_noisy, init_very_noisy]
opt_phens = [opt_noiseless, opt_noisy, opt_very_noisy]
noise_dist = Gamma(1.0,1.0)

average_fits = Matrix{Float64}(undef, 3, max_steps+1)
stds_fits = Matrix{Float64}(undef, 3, max_steps+1)
all_fits = Array{Float64}(undef, 3, number_paths, max_steps+1)

for idx_noise in 1:3
    noisy_list = [all_Ws[idx_noise] .* rand(noise_dist, (number_genes, number_genes))
                  for _ in 1:number_paths]
    dev_paths = explicit_develop.(noisy_list, Ref(init_states[idx_noise]), Ref(max_steps);
        override_stability=true)
    dev_dists = turn_into_distances.(dev_paths, Ref(opt_phens[idx_noise]))

    # (n_steps × number_paths) -> transpose to (number_paths × n_steps)
    fits = permutedims(reduce(hcat, dev_dists))
    fits .= exp.(-params.selection_pressure .* fits)

    all_fits[idx_noise, :, :] .= fits
    average_fits[idx_noise, :] .= vec(mean(fits, dims=1))
    stds_fits[idx_noise, :] .= vec(std(fits, dims=1))
end

In [ ]:
p1 = plot(0:max_steps, average_fits[3,:],
    xlabel="Development Time " * L"(t)", ylabel="Average Fitness",
    label="Evolved with high noise",
    majorgrid=false, minorgrid=false, grid=false,
    ylims = (-0.05, 1.05), color =:green, linewidth=2.0, linestyle=:dash, dpi = 900)
plot!(p1, 0:max_steps, average_fits[2,:], label="Evolved with noise",
    color=:blue, linewidth=2.0, linestyle=:dashdot)
plot!(p1, 0:max_steps, average_fits[1,:], label="Evolved without noise", legend=(0.70, 0.75),
    color =:black, linewidth=2.0)

savefig(p1, "schematic fig 1 c.png")
display(p1)

In [ ]:
ts = 0:max_steps

specs = [
    (1, :black, :solid, "Evolved without noise", "no_noise"),
    (2, :blue, :dashdot, "Evolved with noise", "noise"),
    (3, :green, :dash, "Evolved with high noise", "high_noise"),
]

function make_panel(idx, col, ls, lab; ylabel="Fitness")
    p = plot(xlabel="Development Time " * L"(t)", ylabel=ylabel,
        title=lab, grid=false, majorgrid=false, minorgrid=false,
        ylims=(-0.05, 1.05), legend=false)
    plot!(p, ts, all_fits[idx, :, :]'; color=col, alpha=0.10, linewidth=1.5, label="")
    plot!(p, ts, average_fits[idx, :]; color=col, linestyle=ls, linewidth=2.5, label="Average")
    return p
end

for (idx, col, ls, lab, tag) in specs
    savefig(make_panel(idx, col, ls, lab), "full fig appendix - $tag.png")
end

panels = [make_panel(idx, col, ls, lab; ylabel=idx == 1 ? "Fitness" : "")
          for (idx, col, ls, lab, _) in specs]

p_all = plot(panels..., layout=(1, 3), size=(1500, 400),
    left_margin=7Plots.mm, bottom_margin=5Plots.mm)
savefig(p_all, "full fig appendix.png")
display(p_all)